# Opis obrazu z użyciem Azure OpenAI

Ta część notebooka pokazuje, jak przesłać obraz do modelu multimodalnego Azure OpenAI i poprosić go o opis zawartości. Najpierw używamy obrazu z adresu URL, a następnie wykonujemy to samo dla lokalnego pliku graficznego.

In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv

# Load environment variables
if load_dotenv():
    print("Found Azure OpenAI API Base Endpoint: " + os.getenv("AZURE_OPENAI_ENDPOINT"))
else: 
    print("Azure OpenAI API Base Endpoint not found. Have you configured the .env file?")


client = OpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    base_url=os.getenv("AZURE_OPENAI_ENDPOINT")
)


response = client.chat.completions.create(
    model = os.getenv("AZURE_OPENAI_COMPLETION_MODEL"),
    messages = [ 
        {
            "role": "system", 
            "content": "Jesteś pomocnym asystentem." 
        },
        {
            "role": "user", 
            "content": [
	            {
	                "type": "text",
	                "text": "Opisz obraz:"
	            },
	            {
	                "type": "image_url",
	                "image_url": {
                        "url": "https://msftstories.thesourcemediaassets.com/sites/58/2025/02/Low-res-Brad-MZ107524-1-1536x1024.jpg"
                    }
                } 
           ] 
        }
    ]  

)
response_json = json.loads(response.model_dump_json())
print(json.dumps(response_json, indent=4))

Korzystanie z pliku lokalnego

## Opis obrazu z pliku lokalnego

W tej części notebooka definiujemy funkcję, która koduje lokalny obraz do formatu `data URL`, a następnie wysyłamy go do Azure OpenAI, aby model mógł opisać zawartość pliku `sample.jpg`.

In [ ]:
import base64
import json
from mimetypes import guess_type

# Function to encode a local image into data URL 
def local_image_to_data_url(image_path):
    # Guess the MIME type of the image based on the file extension
    mime_type, _ = guess_type(image_path)
    if mime_type is None:
        mime_type = 'application/octet-stream'  # Default MIME type if none is found

    # Read and encode the image file
    with open(image_path, "rb") as image_file:
        base64_encoded_data = base64.b64encode(image_file.read()).decode('utf-8')

    # Construct the data URL
    return f"data:{mime_type};base64,{base64_encoded_data}"

# Example usage
image_path = './img/sample.jpg' 
data_url = local_image_to_data_url(image_path)

response_local = client.chat.completions.create(
    model = os.getenv("AZURE_OPENAI_COMPLETION_MODEL"),
    messages = [ 
        {
            "role": "system", 
            "content": "Jesteś pomocnym asystentem." 
        },
        {
            "role": "user", 
            "content": [
	            {
	                "type": "text",
	                "text": "Opisz obraz:"
	            },
	            {
	                "type": "image_url",
	                "image_url": {
                        "url": data_url
                    }
                } 
           ] 
        }
    ] 

)
response_local_json = json.loads(response_local.model_dump_json())
print(json.dumps(response_local_json, indent=4))